# Métriques de classification — démonstration sur dix clients

Objectif du Chapitre 2 : voir de ses propres yeux une accuracy de 70 % cacher
un recall de 0.

Aucune donnée réelle ici. Dix clients fabriqués à la main suffisent — c'est
justement parce que le jeu est minuscule qu'on peut recompter chaque cas à la
main et vérifier ce que scikit-learn affiche.

Le vrai dataset arrive au Chapitre 3.

## Le jeu de données : trois features, un label

`X` contient les **features** (ce qu'on connaît du client), `y` le **label**
(ce qu'on veut prédire). La convention `X` majuscule / `y` minuscule vient des
mathématiques — une matrice, un vecteur — et se retrouve dans tout scikit-learn.

In [ ]:
import pandas as pd

# Trois features, un label. Sept clients restent, trois partent.
clients = pd.DataFrame({
    "anciennete_mois":   [ 2, 48,  1, 60,  3, 36, 24,  5, 72,  9],
    "facture_mensuelle": [85, 45, 95, 40, 88, 55, 60, 92, 38, 78],
    "contrat_mensuel":   [ 1,  0,  1,  0,  1,  0,  0,  1,  0,  1],
    "churn":             [ 1,  0,  1,  0,  0,  0,  0,  1,  0,  0],
})

X = clients[["anciennete_mois", "facture_mensuelle", "contrat_mensuel"]]
y = clients["churn"]

print(X.shape, y.shape)
print(y.value_counts(normalize=True))

Sortie attendue :

```
(10, 3) (10,)
churn
0    0.7
1    0.3
Name: proportion, dtype: float64
```

Sept clients sur dix restent : 70 % de classe majoritaire. Retenez ce chiffre,
c'est lui qu'on va retrouver déguisé en « bon score ».

## Le modèle le plus bête du monde

Il ne regarde aucune feature. Il répond « personne ne part », toujours.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

# Modèle baseline : « personne ne part jamais ». Zéro apprentissage.
y_pred_idiot = [0] * len(y)

print("Accuracy :", accuracy_score(y, y_pred_idiot))
print(confusion_matrix(y, y_pred_idiot))

Sortie attendue :

```
Accuracy : 0.7
[[7 0]
 [3 0]]
```

La matrice de confusion se lit : ligne = vérité, colonne = prédiction.

- 7 en haut à gauche : les fidèles correctement identifiés (**vrais négatifs**)
- 3 en bas à gauche : les départs manqués (**faux négatifs**)
- colonne de droite entièrement vide : ce modèle n'a **jamais** prédit un départ

Une accuracy de 0,70 sans avoir écrit la moindre règle.

## Mettre des mots sur le désastre

In [ ]:
from sklearn.metrics import classification_report

# zero_division=0 : sans ça, la précision de la classe 1 (aucune prédiction,
# donc 0/0) déclenche un avertissement au lieu d'afficher proprement 0.00
print(classification_report(y, y_pred_idiot, zero_division=0))

Sortie attendue :

```
              precision    recall  f1-score   support

           0       0.70      1.00      0.82         7
           1       0.00      0.00      0.00         3

    accuracy                           0.70        10
   macro avg       0.35      0.50      0.41        10
weighted avg       0.49      0.70      0.58        10
```

La ligne qui compte est celle de la classe `1`. Précision 0,00, recall 0,00,
F1 0,00. **L'accuracy affichait 70 %, la valeur métier est nulle.**

`support` donne le nombre de cas réels par classe : sept fidèles, trois
partants. C'est la colonne qui révèle un déséquilibre d'un coup d'œil.

## Un modèle un peu moins idiot

Pas de machine learning ici non plus — une règle métier écrite à la main.
Elle suffit à montrer ce que le recall change.

In [ ]:
# Règle métier : contrat mensuel + facture élevée = risque.
y_pred_regle = [
    1 if (contrat == 1 and facture > 80) else 0
    for contrat, facture in zip(clients["contrat_mensuel"],
                                clients["facture_mensuelle"])
]

print("Accuracy :", accuracy_score(y, y_pred_regle))
print(confusion_matrix(y, y_pred_regle))
print(classification_report(y, y_pred_regle, zero_division=0))

Sortie attendue :

```
Accuracy : 0.9
[[6 1]
 [0 3]]
              precision    recall  f1-score   support

           0       1.00      0.86      0.92         7
           1       0.75      1.00      0.86         3

    accuracy                           0.90        10
   macro avg       0.88      0.93      0.89        10
weighted avg       0.93      0.90      0.90        10
```

Trois départs sur trois détectés : recall de 1,00. Une fausse alerte :
précision de 0,75. Le F1 arbitre à 0,86.

## Ce qu'il faut retenir

| Modèle | Accuracy | Recall classe 1 | Départs détectés |
|---|---|---|---|
| « personne ne part » | 0,70 | 0,00 | 0 sur 3 |
| Règle contrat + facture | 0,90 | 1,00 | 3 sur 3 |

Vingt points d'accuracy séparent les deux modèles. C'est le **recall** qui
raconte la vraie histoire : le premier ne sert à rien, le second est
actionnable.

Sur le vrai dataset du Chapitre 3, la classe majoritaire pèse 73,5 % au lieu
de 70 %. Le mécanisme est exactement le même — à 7043 clients près.